<a href="https://colab.research.google.com/github/Rimsabano/Automated-Road-Extraction-/blob/main/Automated-road-extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%writefile data_preprocessing.py
import os
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

def load_dataset(image_dir, mask_dir, size=512, max_images=None):
    image_dataset, mask_dataset = [], []

    image_subdirs = sorted(os.listdir(image_dir))

    if max_images:
        image_subdirs = image_subdirs[:max_images]

    for subdir in image_subdirs:
        img_folder = os.path.join(image_dir, subdir)
        mask_folder = os.path.join(mask_dir, subdir)

        image_files = sorted(os.listdir(img_folder))
        mask_files = sorted(os.listdir(mask_folder))

        for img_file, mask_file in zip(image_files, mask_files):
            img_path = os.path.join(img_folder, img_file)
            mask_path = os.path.join(mask_folder, mask_file)

            image = Image.open(img_path).convert("RGB").resize((size, size))
            mask = Image.open(mask_path).convert("L").resize((size, size))

            image_dataset.append(np.array(image) / 255.0)
            mask_dataset.append(np.array(mask) / 255.0)

    return np.array(image_dataset), np.expand_dims(np.array(mask_dataset), axis=-1)


def split_dataset(X, Y, test_size=0.2, val_size=0.1):

    X_train_val, X_test, y_train_val, y_test = train_test_split(
        X, Y, test_size=test_size, random_state=42
    )

    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val, y_train_val,
        test_size=val_size/(1-test_size),
        random_state=42
    )

    return X_train, X_val, X_test, y_train, y_val, y_test

Writing data_preprocessing.py


In [ ]:
%%writefile layer.py
# simplified safe version (prevents crashes)

import tensorflow as tf
from tensorflow.keras.layers import BatchNormalization, ReLU, Conv2D

def DenseDDSSPPLayer(x, filters, rate):
    x = Conv2D(filters, 1, padding="same")(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)
    return x


def DenseDDSSPP(x):
    for i in range(3):
        x = DenseDDSSPPLayer(x, 64, i)
    return x

Writing layer.py


In [ ]:
%%writefile model.py
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, BatchNormalization, Activation, Concatenate, UpSampling2D
from tensorflow.keras.models import Model
from tensorflow.keras.applications import Xception

def DeepLabV3PlusDenseDDSSPP(input_shape=(512,512,3)):

    inputs = Input(shape=input_shape)

    base = Xception(weights="imagenet", include_top=False, input_tensor=inputs)

    x = base.output

    x = Conv2D(256, 3, padding="same")(x)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)

    x = UpSampling2D((4,4))(x)

    x = Conv2D(1, 1, activation="sigmoid")(x)

    return Model(inputs, x)

Writing model.py


In [ ]:
%%writefile requirements.txt

tensorflow==2.15.0
numpy
pillow
scikit-learn
matplotlib
opencv-python

Writing requirements.txt


In [ ]:
!pip install -r requirements.txt

In [ ]:
%%writefile test.py
import tensorflow as tf

from data_preprocessing import (
    load_dataset,
    split_dataset
)

from model import DeepLabV3PlusDenseDDSSPP

from utils import (
    CustomIoU,
    F1Score
)


# =========================
# PATHS
# =========================

IMAGE_DIR = "/content/data/images"
MASK_DIR = "/content/data/masks"

MODEL_PATH = "outputs/model_checkpoint.keras"

IMAGE_SIZE = 512


# =========================
# LOAD DATASET
# =========================

print("Loading Dataset...")

image_dataset, mask_dataset = load_dataset(
    IMAGE_DIR,
    MASK_DIR,
    size=IMAGE_SIZE,
    max_images=None
)


# =========================
# SPLIT DATASET
# =========================

_, _, X_test, _, _, y_test = split_dataset(
    image_dataset,
    mask_dataset
)


# =========================
# LOAD MODEL
# =========================

print("Loading Model...")

model = DeepLabV3PlusDenseDDSSPP(
    input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3)
)

model.load_weights(MODEL_PATH)


# =========================
# COMPILE MODEL
# =========================

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        CustomIoU(),
        F1Score()
    ]
)


# =========================
# EVALUATE
# =========================

print("Evaluating Model...")

results = model.evaluate(
    X_test,
    y_test,
    verbose=1
)

print("\nTest Results")
print("Loss:", results[0])
print("Accuracy:", results[1])
print("IoU:", results[2])
print("F1 Score:", results[3])

Writing test.py


In [53]:
%%writefile train.py
import os
import tensorflow as tf

from tensorflow.keras.callbacks import (
    ModelCheckpoint,
    EarlyStopping,
    ReduceLROnPlateau
)

from data_preprocessing import (
    load_dataset,
    split_dataset
)

from model import DeepLabV3PlusDenseDDSSPP
from utils import CustomIoU, F1Score


# =========================
# PATHS
# =========================

IMAGE_DIR = "/contents/sample_data/content/data/images"
MASK_DIR = "/content/sample_data/content/data/masks"

MODEL_SAVE_PATH = "outputs/model_checkpoint.keras"

IMAGE_SIZE = 512
BATCH_SIZE = 4
EPOCHS = 50


# =========================
# CREATE OUTPUT DIRECTORY
# =========================

os.makedirs("outputs", exist_ok=True)


# =========================
# LOAD DATASET
# =========================

print("Loading dataset...")

image_dataset, mask_dataset = load_dataset(
    IMAGE_DIR,
    MASK_DIR,
    size=IMAGE_SIZE,
    max_images=None
)

print("Dataset Loaded Successfully")

print("Images Shape:", image_dataset.shape)
print("Masks Shape:", mask_dataset.shape)


# =========================
# SPLIT DATASET
# =========================

X_train, X_val, X_test, y_train, y_val, y_test = split_dataset(
    image_dataset,
    mask_dataset
)


# =========================
# BUILD MODEL
# =========================

print("Building Model...")

model = DeepLabV3PlusDenseDDSSPP(
    input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3)
)


# =========================
# OPTIMIZER
# =========================

optimizer = tf.keras.optimizers.Adam(
    learning_rate=1e-4
)


# =========================
# COMPILE MODEL
# =========================

model.compile(
    optimizer=optimizer,
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        CustomIoU(),
        F1Score()
    ]
)

model.summary()


# =========================
# CALLBACKS
# =========================

checkpoint = ModelCheckpoint(
    MODEL_SAVE_PATH,
    monitor="val_loss",
    save_best_only=True,
    verbose=1
)

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=5,
    verbose=1
)


# =========================
# TRAIN MODEL
# =========================

print("Training Started...")

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[
        checkpoint,
        early_stopping,
        reduce_lr
    ],
    verbose=1
)

print("Training Completed")

FileNotFoundError: [Errno 2] No such file or directory

In [ ]:
%%writefile utils.py
import tensorflow as tf
from tensorflow.keras.metrics import Metric
from tensorflow.keras.metrics import Precision, Recall
from tensorflow.keras import backend as K


class CustomIoU(Metric):

    def __init__(self, num_classes=2, name="custom_iou", **kwargs):
        super(CustomIoU, self).__init__(name=name, **kwargs)

        self.num_classes = num_classes

        self.total_cm = self.add_weight(
            name="total_confusion_matrix",
            shape=(num_classes, num_classes),
            initializer="zeros"
        )

    def update_state(self, y_true, y_pred, sample_weight=None):

        y_true = tf.cast(y_true > 0.5, tf.int32)
        y_pred = tf.cast(y_pred > 0.5, tf.int32)

        y_true = tf.reshape(y_true, [-1])
        y_pred = tf.reshape(y_pred, [-1])

        current_cm = tf.math.confusion_matrix(
            y_true,
            y_pred,
            num_classes=self.num_classes,
            dtype=tf.float32
        )

        self.total_cm.assign_add(current_cm)

    def result(self):

        sum_over_row = tf.reduce_sum(self.total_cm, axis=0)
        sum_over_col = tf.reduce_sum(self.total_cm, axis=1)

        true_positives = tf.linalg.diag_part(self.total_cm)

        denominator = (
            sum_over_row +
            sum_over_col -
            true_positives
        )

        iou = tf.math.divide_no_nan(
            true_positives,
            denominator
        )

        return tf.reduce_mean(iou)

    def reset_state(self):

        K.set_value(
            self.total_cm,
            tf.zeros_like(self.total_cm)
        )


class F1Score(Metric):

    def __init__(self, name="f1_score", **kwargs):

        super(F1Score, self).__init__(
            name=name,
            **kwargs
        )

        self.precision = Precision()
        self.recall = Recall()

    def update_state(self, y_true, y_pred, sample_weight=None):

        self.precision.update_state(
            y_true,
            y_pred,
            sample_weight
        )

        self.recall.update_state(
            y_true,
            y_pred,
            sample_weight
        )

    def result(self):

        precision = self.precision.result()
        recall = self.recall.result()

        return 2 * (
            (precision * recall) /
            (precision + recall + K.epsilon())
        )

    def reset_state(self):

        self.precision.reset_state()
        self.recall.reset_state()

Writing utils.py


In [ ]:
!python train.py

Traceback (most recent call last):
  File "/content/train.py", line 10, in <module>
    from data_preprocessing import (
ModuleNotFoundError: No module named 'data_preprocessing'


In [ ]:
!python test.py

Traceback (most recent call last):
  File "/content/test.py", line 3, in <module>
    from data_preprocessing import (
ModuleNotFoundError: No module named 'data_preprocessing'


In [ ]:
%%writefile predict.py
import os
import cv2
import numpy as np
from PIL import Image
import tensorflow as tf

from model import DeepLabV3PlusDenseDDSSPP


# =========================
# SETTINGS
# =========================

IMAGE_SIZE = 512

MODEL_PATH = "outputs/model_checkpoint.keras"

INPUT_IMAGE_PATH = "sample.jpg"

OUTPUT_DIR = "outputs"

OUTPUT_MASK_PATH = os.path.join(
    OUTPUT_DIR,
    "predicted_mask.png"
)

OVERLAY_OUTPUT_PATH = os.path.join(
    OUTPUT_DIR,
    "overlay_output.png"
)


# =========================
# CREATE OUTPUT DIRECTORY
# =========================

os.makedirs(OUTPUT_DIR, exist_ok=True)


# =========================
# LOAD MODEL
# =========================

print("Loading model...")

model = DeepLabV3PlusDenseDDSSPP(
    input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3)
)

model.load_weights(MODEL_PATH)

print("Model loaded successfully")


# =========================
# LOAD IMAGE
# =========================

print("Loading image...")

original_image = Image.open(INPUT_IMAGE_PATH).convert("RGB")

original_size = original_image.size

image = original_image.resize(
    (IMAGE_SIZE, IMAGE_SIZE)
)

image_array = np.array(image) / 255.0

image_array = np.expand_dims(
    image_array,
    axis=0
)


# =========================
# PREDICTION
# =========================

print("Predicting roads...")

prediction = model.predict(image_array)[0]


# =========================
# PROCESS MASK
# =========================

mask = (prediction > 0.5).astype(np.uint8)

mask = mask.squeeze() * 255

mask_image = Image.fromarray(mask)

mask_image = mask_image.resize(original_size)

mask_image.save(OUTPUT_MASK_PATH)

print("Mask saved successfully")


# =========================
# CREATE OVERLAY
# =========================

original_cv = cv2.cvtColor(
    np.array(original_image),
    cv2.COLOR_RGB2BGR
)

mask_cv = cv2.imread(
    OUTPUT_MASK_PATH,
    cv2.IMREAD_GRAYSCALE
)

colored_mask = np.zeros_like(original_cv)

colored_mask[:, :, 1] = mask_cv

overlay = cv2.addWeighted(
    original_cv,
    0.7,
    colored_mask,
    0.3,
    0
)

cv2.imwrite(
    OVERLAY_OUTPUT_PATH,
    overlay
)

print("Overlay image saved successfully")


# =========================
# DONE
# =========================

print("\nPrediction Completed Successfully")
print("Mask Path:", OUTPUT_MASK_PATH)
print("Overlay Path:", OVERLAY_OUTPUT_PATH)

Writing predict.py
